# CS2 (국내반) --- Lecture 7 보충: `Robot` Class 다시 보기

*이 노트북은 채점/제출 대상이 아닙니다. 본 실습 노트북(`lecture07.ipynb`)을
풀기 전에 class와 object의 개념을 익숙한 Hubo 로봇으로 다시 확인해보는
용도입니다.*

## 왜 Colab용 `Robot`을 따로 만들었나?

CS1에서 사용한 `cs1robots.py`는 로봇 세계를 **내 컴퓨터에 창(window)을 띄워서**
그린다. Colab은 원격 서버에서 실행되기 때문에 창을 띄울 화면이 없어서
`cs1robots`를 그대로 사용할 수 없다.

그래서 이 노트북에서는 `cs1robots`의 `Robot` class를 **간단하게 다시 만든
버전**을 사용한다.
- 사용법은 CS1 때와 같다: `Robot(beepers=6)`, `move()`, `turn_left()`,
  `carries_beepers()`, `drop_beeper()`, `pick_beeper()`, ...
- state 변수 이름도 강의 슬라이드와 같다: `_x`, `_y`, `_dir`,
  `_beeper_bag`
- 애니메이션 대신 `show_world()`를 호출하면 그 시점의 세계(로봇, 지나온
  경로, beeper)를 그림으로 보여준다.
- 원본과 달리 벽(wall)은 없고, 세계의 가장자리만 막혀 있다.

**아래 셀을 먼저 실행하자.** 이번에는 `import`해서 쓰는 대신 class 코드가
눈앞에 있으니, 위에서부터 천천히 읽어보자.

In [ ]:
import matplotlib.pyplot as plt

# ------------------------------------------------------------------
# 로봇 세계 (world): 크기, 바닥에 놓인 beeper, 세계에 있는 로봇들
# ------------------------------------------------------------------
_world = {"avenues": 10, "streets": 10, "beepers": {}, "robots": []}

def create_world(avenues=10, streets=10):
    """avenues x streets 크기의 빈 세계를 새로 만든다."""
    _world["avenues"] = avenues
    _world["streets"] = streets
    _world["beepers"] = {}
    _world["robots"] = []

def place_beepers(avenue, street, n=1):
    """(avenue, street) 위치에 beeper n개를 놓는다 (실습 문제 준비용)."""
    key = (avenue, street)
    _world["beepers"][key] = _world["beepers"].get(key, 0) + n


class RobotError(Exception):
    pass


# 방향 순서: E(동) -> N(북) -> W(서) -> S(남)  (turn_left 할 때마다 다음 방향)
_orient_names = ["E", "N", "W", "S"]
_directions = [(1, 0), (0, 1), (-1, 0), (0, -1)]


class Robot(object):
    # modifier: 새 로봇을 만들 때 자동으로 호출되어 state 변수들을 만든다
    def __init__(self, color="gray", orientation="E", beepers=0,
                 avenue=1, street=1):
        self._color = color
        self._dir = _orient_names.index(orientation)  # 0:E, 1:N, 2:W, 3:S
        self._beeper_bag = beepers                    # 가지고 있는 beeper 수
        self._x = avenue                              # 가로 위치 (avenue)
        self._y = street                              # 세로 위치 (street)
        self._trace = [(avenue, street)]              # 지나온 경로 (그림용)
        _world["robots"].append(self)

    # pure function: 로봇을 print 할 때 보이는 문자열
    def __str__(self):
        return ("Robot(" + self._color + ", (" + str(self._x) + "," +
                str(self._y) + "), " + _orient_names[self._dir] +
                ", beepers=" + str(self._beeper_bag) + ")")

    # modifier: 바라보는 방향으로 한 칸 이동
    def move(self):
        if not self.front_is_clear():
            raise RobotError("That move really hurt! (세계 밖으로 나갈 수 없음)")
        xx, yy = _directions[self._dir]
        self._x += xx
        self._y += yy
        self._trace.append((self._x, self._y))

    # modifier: 왼쪽으로 90도 회전
    def turn_left(self):
        self._dir = (self._dir + 1) % 4

    # pure function: beeper를 하나라도 가지고 있으면 True
    def carries_beepers(self):
        return self._beeper_bag > 0

    # pure function: 지금 서 있는 칸에 beeper가 있으면 True
    def on_beeper(self):
        return _world["beepers"].get((self._x, self._y), 0) > 0

    # pure function: 앞 칸으로 이동할 수 있으면 True
    def front_is_clear(self):
        xx, yy = _directions[self._dir]
        nx, ny = self._x + xx, self._y + yy
        return 1 <= nx <= _world["avenues"] and 1 <= ny <= _world["streets"]

    # pure function: 북쪽을 바라보고 있으면 True
    def facing_north(self):
        return self._dir == 1

    # modifier: 가방의 beeper 하나를 지금 칸에 내려놓음
    def drop_beeper(self):
        if not self.carries_beepers():
            raise RobotError("I don't have any beepers. (가방이 비어 있음)")
        self._beeper_bag -= 1
        place_beepers(self._x, self._y, 1)

    # modifier: 지금 칸의 beeper 하나를 주워 가방에 넣음
    def pick_beeper(self):
        if not self.on_beeper():
            raise RobotError("I'm not on a beeper. (이 칸에 beeper가 없음)")
        key = (self._x, self._y)
        _world["beepers"][key] -= 1
        if _world["beepers"][key] == 0:
            del _world["beepers"][key]
        self._beeper_bag += 1

    # (실습 2) 여기에 새로운 함수를 추가해보자
    # ADD ADDITIONAL CODE HERE!


# ------------------------------------------------------------------
# 세계를 그림으로 보여주는 함수 (Robot class의 일부가 아님)
# ------------------------------------------------------------------
def show_world(title=""):
    W, H = _world["avenues"], _world["streets"]
    fig, ax = plt.subplots(figsize=(0.45 * W + 1, 0.45 * H + 1))
    for i in range(1, W + 1):
        ax.plot([i, i], [0.5, H + 0.5], color="#dddddd", zorder=0)
    for j in range(1, H + 1):
        ax.plot([0.5, W + 0.5], [j, j], color="#dddddd", zorder=0)
    ax.add_patch(plt.Rectangle((0.5, 0.5), W, H, fill=False,
                               edgecolor="#444444", linewidth=2))
    for (x, y), n in _world["beepers"].items():
        ax.add_patch(plt.Circle((x, y), 0.3, color="#2b6cb0", zorder=2))
        ax.text(x + 0.3, y + 0.3, str(n), color="#2b6cb0", ha="center",
                va="center", fontsize=9, fontweight="bold", zorder=5)
    arrows = {0: ">", 1: "^", 2: "<", 3: "v"}
    for r in _world["robots"]:
        xs = [p[0] for p in r._trace]
        ys = [p[1] for p in r._trace]
        ax.plot(xs, ys, color=r._color, alpha=0.5, linewidth=2, zorder=1)
        ax.plot(r._x, r._y, marker=arrows[r._dir], markersize=14,
                color=r._color, markeredgecolor="black", zorder=4)
    ax.set_xlim(0.4, W + 0.6)
    ax.set_ylim(0.4, H + 0.6)
    ax.set_xticks(range(1, W + 1))
    ax.set_yticks(range(1, H + 1))
    ax.set_xlabel("avenue")
    ax.set_ylabel("street")
    ax.set_aspect("equal")
    if title:
        ax.set_title(title)
    plt.show()

print("Robot class 준비 완료!")

## 1. `Robot` class 사용하기 (슬라이드 예제)

강의 슬라이드의 예제를 그대로 실행해보자.

- `hubo = Robot(beepers=6)`을 실행하면 `__init__(hubo, 6)`이 호출된다.
  - 즉, 새 객체 `hubo`가 `self` 자리에 들어가고, `__init__` 안의
    `self._beeper_bag = beepers`가 `hubo._beeper_bag = 6`이 된다.
- `hubo.move()`는 `move(hubo)`가 호출되는 것이다.

In [ ]:
create_world()

hubo = Robot(beepers=6)
# __init__(hubo, 6) invoked

for i in range(9):
    hubo.move()
    # move(hubo) invoked

amy = Robot(color="red", beepers=3)
# __init__(amy, 3) invoked

if amy.carries_beepers():
    # carries_beepers(amy) ..
    print("amy는 beeper를 가지고 있다")

print(hubo)
print(amy)
show_world("hubo (gray) and amy (red)")

## 2. `hubo.move()`는 정말 `move(hubo)`일까?

class 안에 정의된 함수는 `Robot.move`처럼 class 이름으로도 꺼낼 수 있다.
`hubo.move()`는 파이썬이 `Robot.move(hubo)`로 바꿔서 실행해주는 것이다.
그래서 **class 안의 모든 함수는 첫 번째 파라미터로 `self`를 받는다.**

아래 셀을 실행해서 두 방법의 결과가 같은지 확인해보자.

In [ ]:
create_world()
hubo = Robot(beepers=6)

hubo.move()          # 보통 쓰는 방법
print(hubo)          # Robot(gray, (2,1), E, beepers=6)

Robot.move(hubo)     # 파이썬이 실제로 하는 일
print(hubo)          # Robot(gray, (3,1), E, beepers=6)

## 3. Object와 Class

`Robot` class는 `hubo`, `amy`와 같은 로봇의 **type**을 정의한 것으로,
state 변수와 함수들로 구성된다:
- **state 변수**: `_x`, `_y`, `_dir`, `_beeper_bag` 등 각 로봇의 상태
  - `__init__` 함수에서 초기화/생성된다.
- **state 변수를 읽고/쓰는 함수**: `move`, `carries_beepers` 등
  - `move`는 state 변수를 **수정**하고 return 값이 없다 → **modifier**
  - `carries_beepers`는 state 변수를 **수정하지 않고** return 값이 있다
    → **pure function**

Object type을 **class**라고 부른다.
- `hubo`, `amy`는 **object**
- `Robot`은 **class**. 즉, `hubo`와 `amy`의 type

같은 class로 만든 object라도 **state 변수는 각자 따로** 가진다. 아래 셀에서
`hubo`만 움직였을 때 `amy`의 state는 어떻게 되는지 확인해보자.

In [ ]:
create_world()
hubo = Robot(beepers=6)
amy = Robot(color="red", beepers=3, avenue=1, street=3)

print(type(hubo))   # <class '__main__.Robot'>
print(type(amy))    # <class '__main__.Robot'>

hubo.move()
hubo.move()
hubo.drop_beeper()

# state 변수를 직접 들여다보기
print("hubo:", hubo._x, hubo._y, hubo._dir, hubo._beeper_bag)   # hubo: 3 1 0 5
print("amy :", amy._x, amy._y, amy._dir, amy._beeper_bag)       # amy : 1 3 0 3

show_world("only hubo moved")

## 4. 실습

### 실습 1

새 세계를 만들고, `hubo`가 beeper 6개를 가지고 출발해서 동쪽으로 한 칸씩
이동하면서 **지나가는 칸마다 beeper를 1개씩 놓도록** 해보자. 가방의 beeper가
다 떨어지면 멈춘다. (`carries_beepers`를 이용)

In [ ]:
create_world()
hubo = Robot(beepers=6)

# ADD ADDITIONAL CODE HERE!


print(hubo)   # Robot(gray, (7,1), E, beepers=0)
show_world("Practice 1")

### 실습 2

맨 위의 `Robot` class 코드 셀에서 `# (실습 2) 여기에 새로운 함수를 추가해보자`
부분에 다음 두 함수를 **class 안에** 추가하고, 그 셀을 **다시 실행**한 뒤 아래
셀을 실행해보자.

1. `turn_right` (**modifier**)
   - 입력 파라미터: `self`
   - 동작: `self`를 오른쪽으로 90도 회전
   - 리턴값: 없음
2. `beeper_count` (**pure function**)
   - 입력 파라미터: `self`
   - 리턴값: `self`가 가지고 있는 beeper의 수

*class 코드를 고친 뒤에는 class 셀을 다시 실행해야 바뀐 내용이 반영되고,
그 전에 만든 object가 아니라 **새로 만든 object**부터 새 함수를 쓸 수 있다.*

In [ ]:
create_world()
hubo = Robot(beepers=4)

hubo.turn_right()
print(hubo)                 # Robot(gray, (1,1), S, beepers=4)
hubo.turn_left()
hubo.turn_left()
print(hubo)                 # Robot(gray, (1,1), N, beepers=4)

hubo.drop_beeper()
print(hubo.beeper_count())  # 3

### 실습 3

`Robot` class의 각 함수가 **modifier**인지 **pure function**인지 표를
채워보자. (이 셀을 더블클릭해서 편집)

| 함수 | modifier / pure function |
|---|---|
| `__init__` | |
| `__str__` | |
| `move` | |
| `turn_left` | |
| `carries_beepers` | |
| `on_beeper` | |
| `front_is_clear` | |
| `facing_north` | |
| `drop_beeper` | |
| `pick_beeper` | |
| `turn_right` (실습 2) | |
| `beeper_count` (실습 2) | |

### 실습 4

beeper가 `(3,1)`, `(5,1)`, `(8,1)`에 놓여 있다. 빈 가방으로 출발한 `hubo`가
동쪽으로 끝까지 가면서 **beeper를 모두 줍도록** 해보자.
(`front_is_clear`, `on_beeper`, `pick_beeper`를 이용)

In [ ]:
create_world()
place_beepers(3, 1)
place_beepers(5, 1, 2)
place_beepers(8, 1)
hubo = Robot()

# ADD ADDITIONAL CODE HERE!


print(hubo)   # Robot(gray, (10,1), E, beepers=4)
show_world("Practice 4")